In [234]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import scipy as sp
%matplotlib qt
from continuum_solver import ContinuumSolver, clean_input
%load_ext autoreload
%autoreload 2

torch.backends.cuda.preferred_linalg_library("magma")
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
# set default datatype of tensors
DTYPE = torch.complex128

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [235]:
def harmonic_oscillator(x, x_max=5):
    return (1/2)*(x - x_max/2)**2

def mirror_cavity(x, g_0, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    a_mirror = 2 * np.cos(dx) + g_0 * np.sin(dx)
    r = (a_mirror - np.sqrt(a_mirror**2 -  4)) / 2
    g_defect = (2 * r - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def cavity(x, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    g_0 = 2 * (2 - np.cos(dx)) / np.sin(dx)
    g_defect = (2 * (2 - np.sqrt(3)) - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

In [236]:
x_max = 2
x_steps = 128
epsilon = 2e-10
k_vals = torch.linspace(0, 2*torch.pi, 3, device=DEVICE)
energies = torch.linspace(0, 2, 1024, device=DEVICE, dtype=DTYPE)

solver_ho = ContinuumSolver(
    lambda x: harmonic_oscillator(x, x_max=x_max),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_mirror = ContinuumSolver(
    lambda x: cavity(x, 0, x_max, x_steps),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)

In [237]:
solver_mirror.loss(0, energies)

tensor([6.2302e+13, 6.2286e+13, 6.2271e+13,  ..., 4.8394e+13, 4.8382e+13,
        4.8370e+13], device='cuda:0', dtype=torch.float64)

In [238]:
solver_mirror.plot_loss(0, energies)

In [239]:
# fig, ax = plt.subplots(figsize=(10, 8))

# x_plot = np.linspace(0, x_max, 32)
# potential = cavity(x_plot, 0, x_max, 32).cpu().numpy()

# ax.plot(x_plot, potential, label="Potential")

# ax.legend()
# ax.grid()
# ax.set_title("Potential")

# ax.set_xlabel("x")
# ax.set_ylabel("Energy")

# plt.show()